In [58]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import mlflow

from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.combine import SMOTETomek

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier, Perceptron
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score, auc, average_precision_score,precision_recall_curve, classification_report, confusion_matrix

import joblib

import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)


In [2]:
df = pd.read_csv("../data/processed data/merged_data_V2.csv")
df.head()

,SWEAT index,K index,Totals totals index,Date,TH,Environmental Stability,Moisture Indices,Convective Potential,Temperature Pressure,Moisture Temperature Profiles
0,91.2,-1.4,24.7,1981-01-01,0,25.8,22.8,0.0,5636,993.98
1,75.7,1.6,30.3,1981-01-02,0,21.5,20.4,0.0,5592,956.13
2,64.0,2.8,37.5,1981-01-03,0,1.9,19.7,-259.7,5636,862.29
3,128.3,17.5,41.6,1981-01-04,0,13.8,23.8,0.0,5581,978.71
4,194.2,23.5,50.6,1981-01-05,0,4.0,28.6,-55.4,5578,965.77


## Step 1: Apply transformations
- Log transform (np.log1p): applied to right-skewed features  
- Reflect + log: applied to left-skewed features (negate, shift, then log1p)


In [3]:
log_cols       = ['SWEAT index']
reflect_cols   = ['K index', 'Moisture Indices']
shift_log_cols = ['Convective Potential']

# removing extreme tail
df['Convective Potential'] = df['Convective Potential'].clip(
    lower=df['Convective Potential'].quantile(0.01),
    upper=df['Convective Potential'].quantile(0.99)
)

for col in log_cols:
    df[col] = np.log1p(df[col])

for col in reflect_cols:
    df[col] = np.log1p(df[col].max() - df[col])

for col in shift_log_cols:
    shift = abs(df[col].min()) + 1
    df[col] = np.log1p(df[col] + shift)

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SWEAT index,11873.0,5.059036,0.604400,1.916923,4.657763,5.225209,5.521861,6.747234
K index,11873.0,3.614582,0.382736,0.000000,3.310543,3.558201,3.947390,4.831509
Totals totals index,11873.0,39.922311,9.308756,-17.900000,35.400000,41.700000,45.800000,69.800000
TH,11873.0,0.197591,0.398199,0.000000,0.000000,0.000000,0.000000,1.000000
Environmental Stability,11873.0,7.400615,11.181217,-31.200000,0.000000,5.200000,13.300000,58.000000
Moisture Indices,11873.0,4.034062,0.338040,0.000000,3.799974,4.117410,4.312141,4.822698
Convective Potential,11873.0,6.506657,1.169499,0.693147,5.839804,5.883356,7.516113,8.637750
Temperature Pressure,11873.0,5749.557736,70.223398,5530.000000,5698.000000,5756.000000,5805.000000,5961.000000
Moisture Temperature Profiles,11873.0,951.183852,40.405018,497.250000,935.810000,960.430000,979.840000,1168.110000


## Step 2: Clip Outlier

- Not Removing the Outliers because it's not a data error, they are just extreme value.

In [5]:
def check_outlier(cols) -> pd.DataFrame:
    result = []
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers_low = (df[col] < lower).sum()
        outliers_high = (df[col] > upper).sum()

        total_pct = ((outliers_low + outliers_high) / len(df)) * 100

        if total_pct > 1:
            if outliers_low > 0 and outliers_high > 0:
                decision = "Clip Both Side"
            elif outliers_low > 0:
                decision = "Clip Left Side"
            else:
                decision = "Clip Right Side"
        else:
            decision = "Ignore"

        result.append({
            "Field": col,
            "Lower Side Outlier": outliers_low,
            "Upper Side Outlier": outliers_high,
            "Total Percentage": round(total_pct, 2),
            "Decision": decision
        })

    return pd.DataFrame(result)

In [6]:
check_outlier(df.columns.drop(['TH','Date']))

,Field,Lower Side Outlier,Upper Side Outlier,Total Percentage,Decision
0,SWEAT index,87,0,0.73,Ignore
1,K index,3,0,0.03,Ignore
2,Totals totals index,423,31,3.82,Clip Both Side
3,Environmental Stability,17,323,2.86,Clip Both Side
4,Moisture Indices,75,0,0.63,Ignore
5,Convective Potential,144,0,1.21,Clip Left Side
6,Temperature Pressure,4,0,0.03,Ignore
7,Moisture Temperature Profiles,488,6,4.16,Clip Both Side


In [7]:
def clip_outlier(cols_to_clip, cols) -> pd.DataFrame:
    for col, side in cols_to_clip.items():
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        if side == "both":
            df[col] = df[col].clip(lower=lower, upper=upper)
        elif side == "left":
            df[col] = df[col].clip(lower=lower)
        elif side == "right":
            df[col] = df[col].clip(upper=upper)

    return check_outlier(cols)

In [8]:
cols_to_clip = {
    'Totals totals index': 'both',
    'Environmental Stability': 'both',
    'Convective Potential': 'left',
    'Moisture Temperature Profiles':'both',
}
clip_outlier(cols_to_clip, df.columns.drop(['TH','Date']))

,Field,Lower Side Outlier,Upper Side Outlier,Total Percentage,Decision
0,SWEAT index,87,0,0.73,Ignore
1,K index,3,0,0.03,Ignore
2,Totals totals index,0,0,0.00,Ignore
3,Environmental Stability,0,0,0.00,Ignore
4,Moisture Indices,75,0,0.63,Ignore
5,Convective Potential,0,0,0.00,Ignore
6,Temperature Pressure,4,0,0.03,Ignore
7,Moisture Temperature Profiles,0,0,0.00,Ignore


In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SWEAT index,11873.0,5.059036,0.604400,1.916923,4.657763,5.225209,5.521861,6.747234
K index,11873.0,3.614582,0.382736,0.000000,3.310543,3.558201,3.947390,4.831509
Totals totals index,11873.0,40.115000,8.727950,19.800000,35.400000,41.700000,45.800000,61.400000
TH,11873.0,0.197591,0.398199,0.000000,0.000000,0.000000,0.000000,1.000000
Environmental Stability,11873.0,7.280687,10.827969,-19.950000,0.000000,5.200000,13.300000,33.250000
Moisture Indices,11873.0,4.034062,0.338040,0.000000,3.799974,4.117410,4.312141,4.822698
Convective Potential,11873.0,6.535383,1.054010,3.325340,5.839804,5.883356,7.516113,8.637750
Temperature Pressure,11873.0,5749.557736,70.223398,5530.000000,5698.000000,5756.000000,5805.000000,5961.000000
Moisture Temperature Profiles,11873.0,953.028152,33.141884,869.765000,935.810000,960.430000,979.840000,1045.885000


## Step 3: Feature Scaling and Modeling

In [10]:
# feature selection
X = df[df.columns.drop(['TH', 'Date'])]
y = df['TH']

In [11]:
models = {
    #Linear
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Ridge Classifier": RidgeClassifier(random_state=42),
    "SGD Classifier": SGDClassifier(max_iter=1000, random_state=42),
    "Perceptron": Perceptron(max_iter=1000, random_state=42),
    #Tree
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Extra Tree": ExtraTreeClassifier(random_state=42),
    #Ensemble
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    #Neighbors
    "KNN": KNeighborsClassifier(),
    #SVM
    "SVC": SVC(probability=True, random_state=42),
    "Linear SVC": LinearSVC(random_state=42),
    "Nu SVC": NuSVC(nu=0.1, probability=True, random_state=42),
    #Naive Bayes
    "Gaussian NB": GaussianNB(),
    "Bernoulli NB": BernoulliNB(),
    #Discriminant Analysis
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    #Neural Net
    "MLP": MLPClassifier(max_iter=500, random_state=42),
    #Boosts
    "XGBoost":   XGBClassifier(random_state=42, eval_metric="logloss", verbosity=0),
    "LightGBM":  LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost":  CatBoostClassifier(random_state=42, verbose=0),
    #Baseline
    "Dummy (majority)": DummyClassifier(strategy="most_frequent", random_state=42),
    "Dummy (stratified)": DummyClassifier(strategy="stratified", random_state=42)
}

In [ ]:
#train - 80% and test - 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
no_proba_models = ["Ridge Classifier", "Linear SVC", "Perceptron", "SGD Classifier"]

In [14]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Thunderstorm Forecasting")

<Experiment: artifact_location='file:///d:/Project/7. Thunderstorm Forecasting/notebook/mlruns/1', creation_time=1776665980455, experiment_id='1', last_update_time=1776665980455, lifecycle_stage='active', name='Thunderstorm Forecasting', tags={}, trace_location=None, workspace='default'>

In [15]:
pipelines_v1 = {}

for name, model in models.items():
    pipelines_v1[name] = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('model',  model)
    ])

results = []

for name, pipeline in pipelines_v1.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                roc_auc = roc_auc_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("scaler", "StandardScaler")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("roc_auc", roc_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "ROC AUC": round(roc_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "ROC AUC": "",
                "Error": e
            })
results_df = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
results_df

,Model,Accuracy,F1,Precision,Recall,ROC AUC,Error
20,MLP,0.7966,0.0082,0.1111,0.0043,0.7696,
17,Bernoulli NB,0.7036,0.4634,0.3606,0.6482,0.7686,
8,Gradient Boosting,0.8021,0.0637,0.4848,0.0341,0.7674,
18,LDA,0.7920,0.0985,0.3418,0.0576,0.7668,
9,Hist Gradient Boosting,0.7949,0.1411,0.4082,0.0853,0.7664,
23,CatBoost,0.7958,0.1594,0.4259,0.0981,0.7653,
16,Gaussian NB,0.6716,0.4701,0.3450,0.7377,0.7610,
0,Logistic Regression,0.7966,0.1039,0.4000,0.0597,0.7606,
22,LightGBM,0.7937,0.1610,0.4087,0.1002,0.7600,
10,AdaBoost,0.8000,0.0326,0.3636,0.0171,0.7564,


In [ ]:
pipelines_v2 = {}

for name, model in models.items():
    pipelines_v2[name] = Pipeline(steps=[
        ('scaler', RobustScaler()),
        ('model',  model)
    ])

results = []

for name, pipeline in pipelines_v2.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                roc_auc = roc_auc_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("scaler", "RobustScaler")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("roc_auc", roc_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "ROC AUC": round(roc_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "ROC AUC": "",
                "Error": e
            })
results_df2 = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
results_df2

,Model,Accuracy,F1,Precision,Recall,ROC AUC,Error
20,MLP,0.7971,0.0082,0.1176,0.0043,0.7688,
17,Bernoulli NB,0.7116,0.4669,0.3676,0.6397,0.7680,
8,Gradient Boosting,0.8025,0.0713,0.5000,0.0384,0.7679,
18,LDA,0.7920,0.0985,0.3418,0.0576,0.7668,
23,CatBoost,0.7958,0.1565,0.4245,0.0959,0.7638,
22,LightGBM,0.7899,0.1352,0.3611,0.0832,0.7621,
16,Gaussian NB,0.6716,0.4701,0.3450,0.7377,0.7610,
0,Logistic Regression,0.7971,0.1041,0.4058,0.0597,0.7607,
9,Hist Gradient Boosting,0.7886,0.1254,0.3429,0.0768,0.7596,
10,AdaBoost,0.8000,0.0326,0.3636,0.0171,0.7564,


### Experiment Summary:

The top 5 models selected based on recall are:
1. Gaussian Naive Bayes
2. Bernoulli Naive Bayes
3. Quadratic Discriminant Analysis (QDA)
4. Nu-Support Vector Classifier (Nu SVC)
5. Extra Trees Classifier

#### Comparison: StandardScaler vs RobustScaler

A comparison was conducted between StandardScaler and RobustScaler for the selected top 5 models:
- For Gaussian NB, QDA, and Nu SVC, both scalers produced identical performance.
- For Bernoulli NB, StandardScaler showed better recall.
- For Extra Trees, RobustScaler performed slightly better.

#### Conclusion

Based on the overall comparison, StandardScaler is selected as the preferred scaling method for this baseline experiment, as it delivers equal or better performance across the majority of the top-performing models.

In [23]:
top5_models = {
    "Gaussian NB + SMOTE": GaussianNB(),
    "Bernoulli NB + SMOTE": BernoulliNB(),
    "QDA + SMOTE": QuadraticDiscriminantAnalysis(),
    "Nu SVC + SMOTE": NuSVC(nu=0.1, probability=True, random_state=42),
    "Extra Tree + SMOTE": ExtraTreeClassifier(random_state=42)
}

In [28]:
pipelines_v3 = {}

for name, model in top5_models.items():
    pipelines_v3[name] = ImbPipeline(steps=[
        ("smote", SMOTE()),
        ('scaler', StandardScaler()),
        ('model',  model)
    ])

results = []

for name, pipeline in pipelines_v3.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                roc_auc = roc_auc_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("scaler", "StandardScaler")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("roc_auc", roc_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "ROC AUC": round(roc_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "ROC AUC": "",
                "Error": e
            })
results_df3 = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
results_df3

,Model,Accuracy,F1,Precision,Recall,ROC AUC,Error
1,Bernoulli NB + SMOTE,0.6808,0.4892,0.3576,0.7740,0.7721,
0,Gaussian NB + SMOTE,0.6320,0.4677,0.3274,0.8188,0.7603,
2,QDA + SMOTE,0.6303,0.4672,0.3265,0.8209,0.7451,
3,Nu SVC + SMOTE,0.6749,0.3588,0.2939,0.4606,0.6853,
4,Extra Tree + SMOTE,0.6623,0.3635,0.2895,0.4883,0.5967,


In [29]:
pipelines_v4 = {}

for name, model in top5_models.items():
    pipelines_v4[name] = ImbPipeline(steps=[
        ("smote", SMOTE()),
        ('scaler', RobustScaler()),
        ('model',  model)
    ])

results = []

for name, pipeline in pipelines_v4.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                roc_auc = roc_auc_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("scaler", "RobustScaler")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("roc_auc", roc_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "ROC AUC": round(roc_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "ROC AUC": "",
                "Error": e
            })
results_df4 = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
results_df4

,Model,Accuracy,F1,Precision,Recall,ROC AUC,Error
1,Bernoulli NB + SMOTE,0.6922,0.4782,0.3594,0.7143,0.7623,
0,Gaussian NB + SMOTE,0.6324,0.4693,0.3282,0.8230,0.7603,
2,QDA + SMOTE,0.6282,0.4639,0.3243,0.8145,0.7441,
3,Nu SVC + SMOTE,0.6712,0.3960,0.3107,0.5458,0.6943,
4,Extra Tree + SMOTE,0.6838,0.4082,0.3237,0.5522,0.6342,


### Experiment Summary:

The top 3 models selected based on recall are:
1. Gaussian Naive Bayes + SMOTE
2. Bernoulli Naive Bayes + SMOTE
3. Quadratic Discriminant Analysis (QDA) + SMOTE

#### Comparison: StandardScaler vs RobustScaler

A comparison was conducted between StandardScaler and RobustScaler for the selected top 3 models:
- For Gaussian NB, Bernoulli NB and QDA StandardScaler is slightly better then RobustScaler.

#### Conclusion

Based on the overall comparison, StandardScaler is selected as the preferred scaling method for this SMOTE experiment.

##### Ensuring data is sorted by time before splitting

In [30]:
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [31]:
pipelines_v1 = {}

for name, model in models.items():
    pipelines_v1[name] = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('model',  model)
    ])

results = []

for name, pipeline in pipelines_v1.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                roc_auc = roc_auc_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name+"_split_change")
            mlflow.log_param("scaler", "StandardScaler")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("roc_auc", roc_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "ROC AUC": round(roc_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "ROC AUC": "",
                "Error": e
            })
results_df = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
results_df

,Model,Accuracy,F1,Precision,Recall,ROC AUC,Error
20,MLP,0.7865,0.0995,0.4058,0.0567,0.7899,
18,LDA,0.7882,0.0871,0.4211,0.0486,0.7864,
8,Gradient Boosting,0.7941,0.0578,0.6000,0.0304,0.7795,
23,CatBoost,0.7827,0.0979,0.3590,0.0567,0.7755,
22,LightGBM,0.7861,0.1119,0.4103,0.0648,0.7747,
16,Gaussian NB,0.6712,0.4997,0.3655,0.7895,0.7732,
19,QDA,0.7103,0.4827,0.3840,0.6498,0.7732,
0,Logistic Regression,0.7865,0.0995,0.4058,0.0567,0.7731,
17,Bernoulli NB,0.7006,0.4706,0.3722,0.6397,0.7694,
9,Hist Gradient Boosting,0.7844,0.1142,0.3929,0.0668,0.7692,


In [32]:
pipelines_v2 = {}

for name, model in models.items():
    pipelines_v1[name] = Pipeline(steps=[
        ('scaler', RobustScaler()),
        ('model',  model)
    ])

results = []

for name, pipeline in pipelines_v1.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                roc_auc = roc_auc_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_prob)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name+"_split_change")
            mlflow.log_param("scaler", "RobustScaler")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("roc_auc", roc_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "ROC AUC": round(roc_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "ROC AUC": "",
                "Error": e
            })
results_df5 = pd.DataFrame(results).sort_values("ROC AUC", ascending=False)
results_df5

,Model,Accuracy,F1,Precision,Recall,ROC AUC,Error
18,LDA,0.7882,0.0871,0.4211,0.0486,0.7864,
20,MLP,0.7844,0.0229,0.2000,0.0121,0.7820,
9,Hist Gradient Boosting,0.7865,0.1090,0.4133,0.0628,0.7792,
8,Gradient Boosting,0.7941,0.0578,0.6000,0.0304,0.7790,
0,Logistic Regression,0.7865,0.0995,0.4058,0.0567,0.7732,
16,Gaussian NB,0.6712,0.4997,0.3655,0.7895,0.7732,
19,QDA,0.7103,0.4827,0.3840,0.6498,0.7732,
22,LightGBM,0.7853,0.1356,0.4167,0.0810,0.7729,
23,CatBoost,0.7844,0.1049,0.3846,0.0607,0.7721,
17,Bernoulli NB,0.7048,0.4669,0.3739,0.6215,0.7689,


### Experiment Summary:

The top 3 models selected based on recall are:
1. Perceptron
2. Gaussian Naive Bayes
3. Quadratic Discriminant Analysis (QDA)

#### Comparison: StandardScaler vs RobustScaler

A comparison was conducted between StandardScaler and RobustScaler for the selected top 3 models:
- For Perceptron, Bernoulli NB and QDA StandardScaler is slightly better then RobustScaler.

#### Comparison: Time-Based Split vs Stratified Split
A comparison was conducted between time-based splitting and stratified splitting (based on the target variable) for the selected top 3 models:
- For Perceptron, Gaussian Naive Bayes and QDA, the time-based split consistently produced slightly better performance compared to the stratified split.


#### Conclusion

Based on the overall comparison, StandardScaler is selected as the preferred scaling method, and a time-based split is chosen as the data splitting strategy for this baseline experiment.

In [33]:
mlflow.set_experiment("Thunderstorm Forecasting Sampling Experiments")

<Experiment: artifact_location='file:///d:/Project/7. Thunderstorm Forecasting/notebook/mlruns/2', creation_time=1777525191582, experiment_id='2', last_update_time=1777525191582, lifecycle_stage='active', name='Thunderstorm Forecasting Sampling Experiments', tags={}, trace_location=None, workspace='default'>

In [34]:
top3_model = {
    "Perceptron": Perceptron(max_iter=1000, random_state=42),
    "Gaussian NB": GaussianNB(),
    "QDA": QuadraticDiscriminantAnalysis(),
}

In [ ]:
pipelines_v6 = {}

for name, model in top3_model.items():
    pipelines_v6[name] = ImbPipeline(steps=[
        ('sampling', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('model',  model)
    ])


results6 = []

for name, pipeline in pipelines_v6.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                pr_auc = average_precision_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
                pr_auc  = auc(recall_curve, precision_curve)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("sampling", "SMOTE")
            mlflow.log_param("test_size", 0.2)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("pr_auc", pr_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results6.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "PR AUC": round(pr_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results6.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "PR AUC": "",
                "Error": e
            })
results_df6 = pd.DataFrame(results6).sort_values("PR AUC", ascending=False)
results_df6


,Model,Accuracy,F1,Precision,Recall,PR AUC,Error
2,QDA,0.6291,0.4863,0.3415,0.8441,0.4210,
1,Gaussian NB,0.6387,0.4941,0.3486,0.8482,0.3985,
0,Perceptron,0.6013,0.3518,0.2658,0.5202,0.2381,


In [48]:
pipelines_v7 = {}

for name, model in top5_models.items():
    pipelines_v7[name] = ImbPipeline(steps=[
        ('sampling', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('model',  model)
    ])


results7 = []

for name, pipeline in pipelines_v7.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            
            y_prob  = pipeline.predict_proba(X_test)[:, 1]
            precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
            pr_auc  = auc(recall_curve, precision_curve)

            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name+"_top5")
            mlflow.log_param("sampling", "SMOTE")

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("pr_auc", pr_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results7.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "PR AUC": round(pr_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results7.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "PR AUC": "",
                "Error": e
            })
results_df7 = pd.DataFrame(results7).sort_values("PR AUC", ascending=False)
results_df7


,Model,Accuracy,F1,Precision,Recall,PR AUC,Error
4,Extra Tree + SMOTE,0.6657,0.3854,0.3120,0.5040,0.4596,
1,Bernoulli NB + SMOTE,0.6648,0.5062,0.3649,0.8259,0.4415,
2,QDA + SMOTE,0.6291,0.4863,0.3415,0.8441,0.4210,
0,Gaussian NB + SMOTE,0.6387,0.4941,0.3486,0.8482,0.3985,
3,Nu SVC + SMOTE,0.6581,0.3967,0.3134,0.5405,0.3127,


In [49]:
top5_models = {
    "Gaussian NB + BorderlineSMOTE": GaussianNB(),
    "Bernoulli NB + BorderlineSMOTE": BernoulliNB(),
    "QDA + BorderlineSMOTE": QuadraticDiscriminantAnalysis(),
    "Nu SVC + BorderlineSMOTE": NuSVC(nu=0.1, probability=True, random_state=42),
    "Extra Tree + BorderlineSMOTE": ExtraTreeClassifier(random_state=42)
}

In [50]:
pipelines_v8 = {}

for name, model in top5_models.items():
    pipelines_v8[name] = ImbPipeline(steps=[
        ('sampling', BorderlineSMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('model',  model)
    ])


results8 = []

for name, pipeline in pipelines_v8.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                pr_auc = average_precision_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
                pr_auc  = auc(recall_curve, precision_curve)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("sampling", "BorderlineSMOTE")

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("pr_auc", pr_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results8.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "PR AUC": round(pr_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results8.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "PR AUC": "",
                "Error": e
            })
results_df8 = pd.DataFrame(results8).sort_values("PR AUC", ascending=False)
results_df8


,Model,Accuracy,F1,Precision,Recall,PR AUC,Error
4,Extra Tree + BorderlineSMOTE,0.6863,0.3888,0.3269,0.4798,0.4574,
1,Bernoulli NB + BorderlineSMOTE,0.6657,0.5050,0.3649,0.8198,0.4400,
2,QDA + BorderlineSMOTE,0.6282,0.4827,0.3397,0.8340,0.4223,
0,Gaussian NB + BorderlineSMOTE,0.6413,0.4947,0.3498,0.8441,0.3957,
3,Nu SVC + BorderlineSMOTE,0.6724,0.3396,0.2924,0.4049,0.2930,


In [51]:
top5_models = {
    "Gaussian NB + SMOTETomek": GaussianNB(),
    "Bernoulli NB + SMOTETomek": BernoulliNB(),
    "QDA + SMOTETomek": QuadraticDiscriminantAnalysis(),
    "Nu SVC + SMOTETomek": NuSVC(nu=0.1, probability=True, random_state=42),
    "Extra Tree + SMOTETomek": ExtraTreeClassifier(random_state=42)
}

In [52]:
pipelines_v9 = {}

for name, model in top5_models.items():
    pipelines_v9[name] = ImbPipeline(steps=[
        ('sampling', SMOTETomek(random_state=42)),
        ('scaler', StandardScaler()),
        ('model',  model)
    ])


results9 = []

for name, pipeline in pipelines_v9.items():
    with mlflow.start_run(run_name=name):
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            if name in no_proba_models:
                pr_auc = average_precision_score(y_test, y_pred)
            else:
                y_prob  = pipeline.predict_proba(X_test)[:, 1]
                precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
                pr_auc  = auc(recall_curve, precision_curve)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall  = recall_score(y_test, y_pred, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("sampling", "SMOTETomek")

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("pr_auc", pr_auc)

            mlflow.sklearn.log_model(pipeline, name=name)
            
            results9.append({
                "Model": name,
                "Accuracy": round(acc, 4),
                "F1": round(f1, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "PR AUC": round(pr_auc, 4),
                "Error": ""
            })
            
        except Exception as e:
            results9.append({
                "Model": name,
                "Accuracy": "",
                "F1": "",
                "Precision": "",
                "Recall": "",
                "PR AUC": "",
                "Error": e
            })
results_df9 = pd.DataFrame(results9).sort_values("PR AUC", ascending=False)
results_df9


,Model,Accuracy,F1,Precision,Recall,PR AUC,Error
4,Extra Tree + SMOTETomek,0.6825,0.4100,0.3342,0.5304,0.4811,
1,Bernoulli NB + SMOTETomek,0.6686,0.5097,0.3681,0.8279,0.4408,
2,QDA + SMOTETomek,0.6307,0.4880,0.3429,0.8462,0.4212,
0,Gaussian NB + SMOTETomek,0.6383,0.4932,0.3480,0.8462,0.3997,
3,Nu SVC + SMOTETomek,0.6863,0.3425,0.3036,0.3927,0.2916,


### Experiment Summary:

The top 2 models selected based on recall are:
1. Gaussian Naive Bayes
2. Quadratic Discriminant Analysis (QDA)

#### Comparison: SMOTE vs SMOTETomek vs BorderlineSMOTE

A comparison was conducted between SMOTE, SMOTETomek and BorderlineSMOTE for the selected top 5 models:
- For Gaussian NB and QDA SMOTETomek is slightly better then other two.


#### Conclusion

Based on the overall comparison, SMOTETomek is selected as the preferred sampling method for this experiment.

##  Step 4: Hyperparameter Tuning

In [61]:
mlflow.set_experiment("Thunderstorm Forecasting Final Hyperparameter Tuning")

selected_models = {
    "QDA": QuadraticDiscriminantAnalysis(),
    "GaussianNB": GaussianNB()
}

param_grids = {
    "QDA": {
        "model__reg_param": [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
    },
    "GaussianNB": {
        "model__var_smoothing": [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
    }
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tuning_results = []
best_pipelines = {}

for name, model in selected_models.items():
    pipeline = ImbPipeline(steps=[
        ('sampling', SMOTETomek(random_state=42)),
        ('scaler',   StandardScaler()),
        ('model',    model)
    ])

    # Grid Search
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring='recall', 
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    with mlflow.start_run(run_name=f"{name}_Tuned"):
        try:
            search.fit(X_train, y_train)

            best_pipeline = search.best_estimator_
            best_pipelines[name] = best_pipeline

            y_pred = best_pipeline.predict(X_test)
            y_prob = best_pipeline.predict_proba(X_test)[:, 1]

            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)

            precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, y_prob)
            pr_auc = auc(recall_curve, precision_curve)

            f1_scores = 2 * (precision_curve * recall_curve) / (precision_curve + recall_curve + 1e-9)
            best_threshold = thresholds[f1_scores[:-1].argmax()]
            y_pred_tuned = (y_prob >= best_threshold).astype(int)

            f1_tuned = f1_score(y_test, y_pred_tuned, zero_division=0)
            precision_tuned = precision_score(y_test, y_pred_tuned, zero_division=0)
            recall_tuned = recall_score(y_test, y_pred_tuned, zero_division=0)

            mlflow.log_param("model", name)
            mlflow.log_param("sampling","SMOTETomek")
            mlflow.log_param("scaler", "StandardScaler")
            mlflow.log_param("best_params", search.best_params_)
            mlflow.log_param("best_threshold", round(best_threshold, 4))

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1", f1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("pr_auc", pr_auc)
            mlflow.log_metric("f1_tuned", f1_tuned)
            mlflow.log_metric("precision_tuned", precision_tuned)
            mlflow.log_metric("recall_tuned", recall_tuned)

            mlflow.sklearn.log_model(best_pipeline, name=f"{name}_tuned")

            tuning_results.append({
                "Model": name,
                "Best Params": search.best_params_,
                "Best Threshold": round(best_threshold, 4),
                "PR AUC": round(pr_auc, 4),
                "F1 (default)": round(f1, 4),
                "F1 (tuned)": round(f1_tuned, 4),
                "Precision (tuned)": round(precision_tuned, 4),
                "Recall (tuned)": round(recall_tuned, 4),
                "Accuracy": round(acc, 4),
            })

            print(f"\n── {name} ──────────────────────────────")
            print(f"Best Params   : {search.best_params_}")
            print(f"Best Threshold: {best_threshold:.4f}")
            print(classification_report(y_test, y_pred_tuned,
                                        target_names=['No Storm', 'Storm']))

            cm = confusion_matrix(y_test, y_pred_tuned)

        except Exception as e:
            print(f"Error in {name}: {e}")
            tuning_results.append({
                "Model": name, "Error": str(e)
            })

tuning_df = pd.DataFrame(tuning_results).sort_values("PR AUC", ascending=False)
tuning_df

Fitting 5 folds for each of 6 candidates, totalling 30 fits

── QDA ──────────────────────────────
Best Params   : {'model__reg_param': 0.3}
Best Threshold: 0.7264
              precision    recall  f1-score   support

    No Storm       0.91      0.69      0.78      1881
       Storm       0.38      0.73      0.50       494

    accuracy                           0.70      2375
   macro avg       0.64      0.71      0.64      2375
weighted avg       0.80      0.70      0.72      2375

Fitting 5 folds for each of 7 candidates, totalling 35 fits

── GaussianNB ──────────────────────────────
Best Params   : {'model__var_smoothing': 1e-11}
Best Threshold: 0.6628
              precision    recall  f1-score   support

    No Storm       0.93      0.61      0.74      1881
       Storm       0.36      0.83      0.50       494

    accuracy                           0.65      2375
   macro avg       0.65      0.72      0.62      2375
weighted avg       0.81      0.65      0.69      2375



,Model,Best Params,Best Threshold,PR AUC,F1 (default),F1 (tuned),Precision (tuned),Recall (tuned),Accuracy
0,QDA,{'model__reg_param': 0.3},0.7264,0.4235,0.4928,0.5000,0.3795,0.7328,0.6291
1,GaussianNB,{'model__var_smoothing': 1e-11},0.6628,0.3997,0.4932,0.5006,0.3580,0.8320,0.6383


## Step 5: Conclusion

The objective of this project was to build a classification model capable of predicting storm events using machine learning techniques. After extensive experimentation across multiple models, preprocessing strategies, and hyperparameter tuning, QDA (Quadratic Discriminant Analysis) emerged as the best performing model.